# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdelrhman-Moubarak/FlyRank-ML-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Setup

In [2]:
import duckdb
import pandas as pd
from dotenv import load_dotenv
import os

load_dotenv()
hf_token = os.getenv('HF_TOKEN')

con = duckdb.connect()
con.sql(f"CREATE SECRET hf_token (TYPE huggingface, TOKEN '{hf_token}');")

FACT_MARCH = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
DIM_CONTENT = 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'

## 1. Method choice and why

**Why it fits:** the label is imbalanced, not close to 50/50, so I need class_weight="balanced". The features mix a category (content_type) with numbers on very different scales (age in days, impressions in the thousands), and one feature is an interaction term (age_x_impressions). A tree-based model splits on raw cutoffs, so it does not need scaling and can pick up the interaction on its own. The code below checks the class balance and feature mix this choice is based on.

In [3]:
#########################################################
print("=== BUILDING THE FRAME (H1 features + H2 label) ===")

h1_full = con.sql(f"""
    WITH h1 AS (
        SELECT
            content_hash_id,
            client_hash_id,
            AVG(gsc_avg_position)   AS average_search_position,
            SUM(gsc_clicks)         AS total_clicks_h1,
            SUM(gsc_impressions)    AS total_impressions_h1
        FROM read_parquet('{FACT_MARCH}')
        WHERE report_date BETWEEN '2026-03-01' AND '2026-03-15'
        GROUP BY content_hash_id, client_hash_id
    )
    SELECT
        h1.*,
        DATE_DIFF('day', dc.content_created_date, DATE '2026-03-01') AS content_age_days,
        dc.content_type
    FROM h1
    JOIN read_parquet('{DIM_CONTENT}') dc USING (content_hash_id)
""").df()

label_h2 = con.sql(f"""
    SELECT content_hash_id, client_hash_id, SUM(gsc_clicks) AS total_clicks_h2
    FROM read_parquet('{FACT_MARCH}')
    WHERE report_date BETWEEN '2026-03-16' AND '2026-03-31'
    GROUP BY content_hash_id, client_hash_id
""").df()

frame = h1_full.merge(label_h2, on=["content_hash_id", "client_hash_id"], how="inner")
frame = frame.sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)

H1_DAYS = 15
H2_DAYS = 16
frame["is_declining"] = (frame["total_clicks_h2"] / H2_DAYS < frame["total_clicks_h1"] / H1_DAYS).astype(int)
frame["has_impressions"] = (frame["total_impressions_h1"] > 0).astype(int)

print("shape:", frame.shape)

#########################################################
print("\n=== WHY RANDOM FOREST FITS THIS LANE ===")

decline_rate = frame["is_declining"].mean()
n_content_types = frame["content_type"].nunique()

print("decline rate (class balance):", decline_rate)
print("distinct content_type values:", n_content_types)
print("age range in days, min:", frame["content_age_days"].min(), "max:", frame["content_age_days"].max())
print("total_impressions_h1 range, min:", frame["total_impressions_h1"].min(), "max:", frame["total_impressions_h1"].max())
print("this confirms an imbalanced label and a mix of category and wide-range numeric features")

=== BUILDING THE FRAME (H1 features + H2 label) ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

shape: (319758, 10)

=== WHY RANDOM FOREST FITS THIS LANE ===
decline rate (class balance): 0.11391427266870571
distinct content_type values: 3
age range in days, min: -19 max: 464
total_impressions_h1 range, min: 0.0 max: 161575.0
this confirms an imbalanced label and a mix of category and wide-range numeric features


## 2. Split design

Split: `GroupShuffleSplit` grouped by `client_hash_id`, 80/20, `random_state=42`. Not a random row split, since pages from the same client behave alike (same industry, same content team). A random split would let the model see a client's pattern in training and win easily on that client's other pages in test. Grouping by client keeps each client on one side only, so the test score reflects unseen clients.

Three more choices in the feature code:

- **Rounding `avg_position_h1` to 6 decimals:** zero-impression pages all get the same imputed median position, but floating point noise made some "identical" values compare unequal, undercounting duplicate rows in a later check. Rounding fixes that.
- **`total_impressions_h1` vs `has_impressions`:** about half the pages have zero impressions and share the same imputed position. The flag lets the model tell a true zero-visibility page apart from one that just sits near the median on its own.
- **`age_x_impressions`:** age and content type alone barely predict decline. Old content that still pulls impressions behaves differently from old content that has faded, so this interaction captures that, and it becomes the model's second most important feature.

In [4]:
#########################################################
print("=== FEATURE ENGINEERING ===")

median_position = frame.loc[frame["has_impressions"] == 1, "average_search_position"].median()
frame["avg_position_h1"] = frame["average_search_position"]
frame.loc[frame["has_impressions"] == 0, "avg_position_h1"] = median_position
frame["avg_position_h1"] = frame["avg_position_h1"].round(6)

frame["age_x_impressions"] = frame["content_age_days"] * frame["total_impressions_h1"]

print("median position used for imputation:", median_position)
print("nulls remaining in avg_position_h1:", frame["avg_position_h1"].isna().sum())

#########################################################
print("\n=== BUILD MODEL FRAME ===")

feature_cols = ["content_age_days", "content_type", "total_impressions_h1", "has_impressions", "avg_position_h1", "age_x_impressions"]
model_frame = frame[feature_cols + ["is_declining", "client_hash_id", "content_hash_id"]].dropna(subset=["content_age_days"])
model_frame = model_frame[model_frame["content_age_days"] >= 0]
model_frame = pd.get_dummies(model_frame, columns=["content_type"], drop_first=True)

print("model_frame shape:", model_frame.shape)
print("distinct clients:", model_frame["client_hash_id"].nunique())

#########################################################
print("\n=== CLIENT-GROUPED SPLIT ===")

from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(model_frame, groups=model_frame["client_hash_id"]))
train, test = model_frame.iloc[train_idx], model_frame.iloc[test_idx]

overlap_clients = set(train["client_hash_id"]) & set(test["client_hash_id"])

print("train shape:", train.shape)
print("test shape:", test.shape)
print("clients in train:", train["client_hash_id"].nunique())
print("clients in test:", test["client_hash_id"].nunique())
print("clients overlapping train and test:", len(overlap_clients))
print("train decline rate:", train["is_declining"].mean())
print("test decline rate:", test["is_declining"].mean())

=== FEATURE ENGINEERING ===
median position used for imputation: 8.285743145743147
nulls remaining in avg_position_h1: 0

=== BUILD MODEL FRAME ===
model_frame shape: (302692, 10)
distinct clients: 52

=== CLIENT-GROUPED SPLIT ===
train shape: (249445, 10)
test shape: (53247, 10)
clients in train: 41
clients in test: 11
clients overlapping train and test: 0
train decline rate: 0.11765719898173946
test decline rate: 0.10612804477247545


## 3. Train + compare vs my baseline

Same split, same metric (Precision@50) as the baseline. I picked hyperparameters with 5-fold `GroupKFold` cross validation, grid search over `max_depth` and `min_samples_leaf`, scored by mean minus standard deviation, not just the highest average, so the choice favors a config that stays stable across folds rather than one that got lucky on one fold. That picked `max_depth=8, min_samples_leaf=20`.

| method | precision | lift vs random | lift vs baseline |
|---|---|---|---|
| Random chance | 0.106 | 1.00x | 0.70x |
| Baseline rule (refresh tier) | 0.153 | 1.44x | 1.00x |
| Model, single test split | 0.38 | 3.58x | 2.49x |
| Model, CV mean | 0.652 | 6.14x | 4.27x |

The model beats the baseline by about 2.5x on the same split, same metric, and the CV mean (0.652) is well above the single split score, which is expected since Precision@50 on one 53k-row test set has real fold-to-fold noise, shown by the CV std of 0.085.

In [5]:
#########################################################
print("=== BUILD BASELINE RULE (ML-07, REBUILT ON THIS SPLIT) ===")

rule_frame = frame[frame["content_age_days"] >= 0].copy()
rule_frame["age_bucket"] = pd.qcut(rule_frame["content_age_days"], q=3, labels=["young", "mid", "old"])
age_points = {"young": 3, "mid": 1, "old": 0}

type_points = {"keyword article": 2, "comparison article": 1, "feedly article": 0}

rule_frame["age_points"] = rule_frame["age_bucket"].map(age_points)
rule_frame["type_points"] = rule_frame["content_type"].map(type_points)
rule_frame["score"] = rule_frame["age_points"].astype(float) + rule_frame["type_points"].astype(float)

rule_frame["action"] = "no_action"
rule_frame.loc[rule_frame["score"] >= 2, "action"] = "monitor"
rule_frame.loc[rule_frame["score"] >= 4, "action"] = "refresh"

rule_test = rule_frame[rule_frame["content_hash_id"].isin(test["content_hash_id"])]
refresh_tier = rule_test[rule_test["action"] == "refresh"]

print("refresh tier size:", len(refresh_tier))
print("refresh tier precision:", refresh_tier["is_declining"].mean())
print("random guessing rate on this split:", rule_test["is_declining"].mean())

#########################################################
print("\n=== HYPERPARAMETER TUNING: 5-FOLD GroupKFold CROSS VALIDATION ===")

from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
import numpy as np

def precision_at_50(mdl, X_tr, y_tr, X_te, y_te, ids_te):
    mdl.fit(X_tr, y_tr)
    probs = mdl.predict_proba(X_te)[:, 1]
    res = pd.DataFrame({"content_hash_id": ids_te, "is_declining": y_te.values, "predicted_prob": probs})
    res = res.sort_values(["predicted_prob", "content_hash_id"], ascending=[False, True]).reset_index(drop=True)
    return res.head(50)["is_declining"].mean()

drop_cols = ["is_declining", "client_hash_id", "content_hash_id"]
X_all = model_frame.drop(columns=drop_cols)
y_all = model_frame["is_declining"]
groups_all = model_frame["client_hash_id"]
ids_all = model_frame["content_hash_id"]

gkf = GroupKFold(n_splits=5)
max_depth_grid = [3, 5, 8, 12, None]
min_leaf_grid = [1, 5, 20, 50]

sweep_results = []
for max_depth in max_depth_grid:
    for min_leaf in min_leaf_grid:
        fold_scores = []
        for tr_idx, te_idx in gkf.split(X_all, y_all, groups=groups_all):
            X_tr = X_all.iloc[tr_idx]
            y_tr = y_all.iloc[tr_idx]
            X_te = X_all.iloc[te_idx]
            y_te = y_all.iloc[te_idx]
            ids_te = ids_all.iloc[te_idx]
            sweep_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced",
                                                  n_jobs=-1, max_depth=max_depth, min_samples_leaf=min_leaf)
            score = precision_at_50(sweep_model, X_tr, y_tr, X_te, y_te, ids_te)
            fold_scores.append(score)
        sweep_results.append({"max_depth": max_depth, "min_samples_leaf": min_leaf,
                               "mean_precision_at_50": np.mean(fold_scores), "std_precision_at_50": np.std(fold_scores)})

sweep_df = pd.DataFrame(sweep_results)
sweep_df["mean_minus_std"] = sweep_df["mean_precision_at_50"] - sweep_df["std_precision_at_50"]
sweep_df = sweep_df.sort_values("mean_minus_std", ascending=False).reset_index(drop=True)
print(sweep_df)

#########################################################
print("\n=== PICK FINAL CONFIG: BEST MEAN MINUS STD ===")

chosen_row = sweep_df.iloc[[0]]
chosen_max_depth = chosen_row["max_depth"].values[0]
chosen_min_leaf = int(chosen_row["min_samples_leaf"].values[0])
if pd.isna(chosen_max_depth):
    chosen_max_depth = None
else:
    chosen_max_depth = int(chosen_max_depth)

print("chosen max_depth:", chosen_max_depth)
print("chosen min_samples_leaf:", chosen_min_leaf)
print(chosen_row[["mean_precision_at_50", "std_precision_at_50", "mean_minus_std"]])

#########################################################
print("\n=== TRAIN FINAL MODEL ON THE CLIENT-GROUPED SPLIT ===")

X_train, y_train = train.drop(columns=drop_cols), train["is_declining"]
X_test, y_test = test.drop(columns=drop_cols), test["is_declining"]

model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced",
                                n_jobs=1, max_depth=chosen_max_depth, min_samples_leaf=chosen_min_leaf)
model.fit(X_train, y_train)
test_probs = model.predict_proba(X_test)[:, 1]

results_frame = X_test.copy()
results_frame["is_declining"] = y_test.values
results_frame["predicted_prob"] = test_probs
results_frame["content_hash_id"] = test["content_hash_id"].values
results_frame = results_frame.sort_values(["predicted_prob", "content_hash_id"], ascending=[False, True]).reset_index(drop=True)

top50_model = results_frame.head(50)
precision_at_50_model = top50_model["is_declining"].mean()

print("distinct predicted probabilities:", results_frame["predicted_prob"].nunique())
print("model precision at 50:", precision_at_50_model)

#########################################################
print("\n=== MODEL VS BASELINE, SAME SPLIT, SAME METRIC ===")

baseline_precision = refresh_tier["is_declining"].mean()
random_rate = rule_test["is_declining"].mean()

comparison_table = pd.DataFrame([
    {"method": "Random chance", "precision": random_rate, "lift_vs_random": 1.0},
    {"method": "Baseline rule (refresh tier)", "precision": baseline_precision, "lift_vs_random": baseline_precision / random_rate},
    {"method": "Model, single test split", "precision": precision_at_50_model, "lift_vs_random": precision_at_50_model / random_rate},
    {"method": "Model, CV mean", "precision": chosen_row["mean_precision_at_50"].values[0], "lift_vs_random": chosen_row["mean_precision_at_50"].values[0] / random_rate},
])
comparison_table["lift_vs_baseline"] = comparison_table["precision"] / baseline_precision

print(comparison_table.to_string(index=False))
print("\nlift of model over baseline rule:", precision_at_50_model / baseline_precision)

=== BUILD BASELINE RULE (ML-07, REBUILT ON THIS SPLIT) ===
refresh tier size: 14561
refresh tier precision: 0.15266808598310555
random guessing rate on this split: 0.10612804477247545

=== HYPERPARAMETER TUNING: 5-FOLD GroupKFold CROSS VALIDATION ===
    max_depth  min_samples_leaf  mean_precision_at_50  std_precision_at_50  \
0         8.0                20                 0.652             0.085417   
1         8.0                 5                 0.668             0.114961   
2         NaN                20                 0.600             0.052154   
3         8.0                50                 0.620             0.076942   
4         8.0                 1                 0.608             0.080598   
5         3.0                20                 0.576             0.074189   
6         3.0                 5                 0.576             0.074189   
7         3.0                 1                 0.576             0.074189   
8         3.0                50                

## 4. Errors and interpretation

Feature importance shows the model leans almost entirely on search performance, not on the two features the baseline rule uses: `total_impressions_h1` and `age_x_impressions` together account for about 70% of the model's importance, while `content_age_days` and `content_type` barely matter on their own, even though they looked useful in the earlier bucket checks.

The real ceiling on this model is a collision problem, not a modeling failure. 55.1% of rows share an identical feature row, mostly because the 52.2% of pages with zero impressions in H1 all get the same imputed position, making them indistinguishable to the model no matter how it is tuned.

I also tested adding back the four excluded columns (`search_volume`, `backlinks`, `word_count`, `competition`). This looked like it raised Precision@50 from 0.38 to 0.68, but that jump came entirely from testing on a smaller, easier subset of pages (113,463 rows instead of 302,692, since those columns have missing values), not from the columns adding real signal. Proof: the original model, with no extra columns, scored even higher (0.70) on that same restricted subset.

In [6]:
#########################################################
print("=== FEATURE IMPORTANCE ===")

importance_table = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False).reset_index(drop=True)

print(importance_table.to_string(index=False))
print("\nthe baseline rule uses content_age_days and content_type only")
print("the model barely uses those two on their own, and leans on impressions and the age-impressions interaction instead")

#########################################################
print("\n=== ERROR ANALYSIS: FEATURE COLLISION ===")

feature_cols_only = [c for c in model_frame.columns if c not in ("client_hash_id", "content_hash_id")]
dupe_pct = model_frame.duplicated(subset=feature_cols_only).mean() * 100
zero_impr_pct = (model_frame["has_impressions"] == 0).mean() * 100

print("rows with an exact duplicate feature row:", round(dupe_pct, 1), "%")
print("zero-impression rows (all get the same imputed position):", round(zero_impr_pct, 1), "%")
print("this collision, not the model or the tuning, is the real ceiling on precision")

#########################################################
print("\n=== ERROR ANALYSIS: EXCLUDED COLUMNS, A NEGATIVE RESULT ===")

extra_cols_test = con.sql(f"""
    SELECT content_hash_id, search_volume, backlinks, word_count, competition
    FROM read_parquet('{DIM_CONTENT}')
""").df()

model_frame_extra = frame[feature_cols + ["is_declining", "client_hash_id", "content_hash_id"]].dropna(subset=["content_age_days"])
model_frame_extra = model_frame_extra[model_frame_extra["content_age_days"] >= 0]
model_frame_extra = model_frame_extra.merge(extra_cols_test, on="content_hash_id", how="left")
model_frame_extra = model_frame_extra.dropna(subset=["search_volume", "backlinks", "word_count", "competition"])
model_frame_extra = pd.get_dummies(model_frame_extra, columns=["content_type"], drop_first=True)

gss_extra = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx_extra, test_idx_extra = next(gss_extra.split(model_frame_extra, groups=model_frame_extra["client_hash_id"]))
train_extra, test_extra = model_frame_extra.iloc[train_idx_extra], model_frame_extra.iloc[test_idx_extra]

extra_only_cols = ["search_volume", "backlinks", "word_count", "competition"]
original_cols_only = [c for c in model_frame_extra.columns if c not in drop_cols and c not in extra_only_cols]

# Model A: extra features, restricted population
X_train_extra = train_extra.drop(columns=drop_cols)
y_train_extra = train_extra["is_declining"]
X_test_extra = test_extra.drop(columns=drop_cols)
y_test_extra = test_extra["is_declining"]

model_extra = RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced",
                                      n_jobs=1, max_depth=chosen_max_depth, min_samples_leaf=chosen_min_leaf)
model_extra.fit(X_train_extra, y_train_extra)
probs_extra = model_extra.predict_proba(X_test_extra)[:, 1]
results_extra = X_test_extra.copy()
results_extra["is_declining"] = y_test_extra.values
results_extra["predicted_prob"] = probs_extra
results_extra = results_extra.sort_values("predicted_prob", ascending=False).reset_index(drop=True)
precision_at_50_extra = results_extra.head(50)["is_declining"].mean()

# Model B: original features only, same restricted population
X_train_restricted = train_extra[original_cols_only]
X_test_restricted = test_extra[original_cols_only]

model_restricted = RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced",
                                           n_jobs=1, max_depth=chosen_max_depth, min_samples_leaf=chosen_min_leaf)
model_restricted.fit(X_train_restricted, y_train_extra)
probs_restricted = model_restricted.predict_proba(X_test_restricted)[:, 1]
results_restricted = X_test_restricted.copy()
results_restricted["is_declining"] = y_test_extra.values
results_restricted["predicted_prob"] = probs_restricted
results_restricted = results_restricted.sort_values("predicted_prob", ascending=False).reset_index(drop=True)
precision_restricted_original_features = results_restricted.head(50)["is_declining"].mean()

print("rows, full population:", len(model_frame))
print("rows, restricted to pages with all four extra columns present:", len(model_frame_extra))
print()
print("precision at 50, original features, full population:", precision_at_50_model)
print("precision at 50, original features, restricted population:", precision_restricted_original_features)
print("precision at 50, extra features, restricted population:", precision_at_50_extra)
print()
print("the jump comes from testing on an easier, restricted population, not from the extra columns")
print("proof: the original model, no extra columns, scores just as high on that same restricted population")

=== FEATURE IMPORTANCE ===
                     feature  importance
        total_impressions_h1    0.412165
           age_x_impressions    0.290401
             has_impressions    0.164356
             avg_position_h1    0.082338
content_type_keyword article    0.023083
            content_age_days    0.021315
 content_type_feedly article    0.006343

the baseline rule uses content_age_days and content_type only
the model barely uses those two on their own, and leans on impressions and the age-impressions interaction instead

=== ERROR ANALYSIS: FEATURE COLLISION ===
rows with an exact duplicate feature row: 55.1 %
zero-impression rows (all get the same imputed position): 52.2 %
this collision, not the model or the tuning, is the real ceiling on precision

=== ERROR ANALYSIS: EXCLUDED COLUMNS, A NEGATIVE RESULT ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows, full population: 302692
rows, restricted to pages with all four extra columns present: 113463

precision at 50, original features, full population: 0.38
precision at 50, original features, restricted population: 0.7
precision at 50, extra features, restricted population: 0.68

the jump comes from testing on an easier, restricted population, not from the extra columns
proof: the original model, no extra columns, scores just as high on that same restricted population


Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.